# C4-classical-ml-practice — Practice p16 — Solution

In [ ]:
import numpy as np
import pandas as pd

SEED = 20260804
FEATURES = ["temp_c", "vibration_mm_s", "pressure_pa"]
sensors = pd.read_csv("data/sensors.csv")
Xs = sensors[FEATURES].to_numpy()
ys = sensors["status"].to_numpy()
q = Xs[0]
X_rest, y_rest = Xs[1:], ys[1:]
raw_d2 = ((X_rest - q) ** 2).sum(axis=1)
nn5_raw_labels = y_rest[np.argsort(raw_d2)[:5]]
raw_values, raw_counts = np.unique(nn5_raw_labels, return_counts=True)
vote_raw = raw_values[np.argmax(raw_counts)]
mu = X_rest.mean(axis=0)
col_stds = X_rest.std(axis=0)
Z_rest = (X_rest - mu) / col_stds
z_q = (q - mu) / col_stds
scaled_d2 = ((Z_rest - z_q) ** 2).sum(axis=1)
nn5_scaled_labels = y_rest[np.argsort(scaled_d2)[:5]]
scaled_values, scaled_counts = np.unique(nn5_scaled_labels, return_counts=True)
vote_scaled = scaled_values[np.argmax(scaled_counts)]

nn5_raw_labels, vote_raw, nn5_scaled_labels, vote_scaled, col_stds

The training standard deviations are about 2.406 °C, 1.185 mm/s, and 1168.94 Pa, so unscaled pressure differences numerically overwhelm the other coordinates and radically change the neighbor ordering. Uri’s scaled list is the one to trust: Vera ignores scale dependence, and the nearly equal class-mean pressures (about 100523 versus 100774 Pa) show that raw pressure domination is mostly noise; the scaled list’s five `ok` neighbors is also cleaner than Tamar’s mixed 3–2 list.

### Answer check

In [ ]:
assert np.array_equal(nn5_raw_labels, np.array(["ok", "faulty", "ok", "faulty", "ok"]))
assert vote_raw == "ok"
assert np.array_equal(nn5_scaled_labels, np.array(["ok", "ok", "ok", "ok", "ok"]))
assert vote_scaled == "ok"
assert np.allclose(col_stds, np.array([2.405674823995325, 1.1853223542972045, 1168.9397484429214]), atol=1e-9, rtol=0)
assert sensors.loc[0, "status"] == vote_scaled